# 💧 LFM2.5-350M In-Notebook Chatbot

This notebook loads [LiquidAI/LFM2.5-350M-ONNX](https://huggingface.co/LiquidAI/LFM2.5-350M-ONNX) and runs a **chat UI entirely inside Colab** using `google.colab.kernel.invokeFunction` to bridge JS → Python.

No Flask, no tunnels — just direct kernel calls.

## 1 — Install dependencies

In [ ]:
!pip install -q onnxruntime transformers huggingface_hub numpy

## 2 — Download model & tokenizer

In [ ]:
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer

MODEL_ID = "LiquidAI/LFM2.5-350M-ONNX"
VARIANT = "model_q4"  # smallest variant (~276MB)

print("Downloading ONNX model...")
model_path = hf_hub_download(MODEL_ID, f"onnx/{VARIANT}.onnx")
data_path  = hf_hub_download(MODEL_ID, f"onnx/{VARIANT}.onnx_data")

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print(f"Model downloaded to: {model_path}")

## 3 — Load ONNX session & define generation

In [ ]:
import numpy as np
import onnxruntime as ort

print("Loading ONNX session (this may take a moment)...")
session = ort.InferenceSession(model_path)
print(f"Session loaded: {len(session.get_inputs())} inputs, {len(session.get_outputs())} outputs")

TEMPERATURE = 0.1
TOP_K = 50
REPETITION_PENALTY = 1.05
MAX_NEW_TOKENS = 512

ONNX_DTYPE = {
    "tensor(float)": np.float32,
    "tensor(float16)": np.float16,
    "tensor(int64)": np.int64,
}

input_names = {inp.name for inp in session.get_inputs()}
use_position_ids = "position_ids" in input_names


def _init_cache():
    cache = {}
    for inp in session.get_inputs():
        if inp.name in {"input_ids", "attention_mask", "position_ids"}:
            continue
        shape = [d if isinstance(d, int) else 1 for d in inp.shape]
        for i, d in enumerate(inp.shape):
            if isinstance(d, str) and "sequence" in d.lower():
                shape[i] = 0
        cache[inp.name] = np.zeros(shape, dtype=ONNX_DTYPE.get(inp.type, np.float32))
    return cache


def _sample_token(logits, generated_tokens):
    for tid in set(generated_tokens):
        if logits[tid] > 0:
            logits[tid] /= REPETITION_PENALTY
        else:
            logits[tid] *= REPETITION_PENALTY
    logits = logits / TEMPERATURE
    top_k_idx = np.argpartition(logits, -TOP_K)[-TOP_K:]
    top_k_logits = logits[top_k_idx]
    top_k_logits -= np.max(top_k_logits)
    probs = np.exp(top_k_logits) / np.sum(np.exp(top_k_logits))
    chosen = np.random.choice(len(top_k_idx), p=probs)
    return int(top_k_idx[chosen])


def generate(messages: list) -> str:
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    input_ids = np.array(
        [tokenizer.encode(prompt, add_special_tokens=False)], dtype=np.int64
    )
    seq_len = input_ids.shape[1]
    cache = _init_cache()
    generated_tokens = []

    for step in range(MAX_NEW_TOKENS):
        if step == 0:
            ids = input_ids
            pos = np.arange(seq_len, dtype=np.int64).reshape(1, -1)
        else:
            ids = np.array([[generated_tokens[-1]]], dtype=np.int64)
            pos = np.array([[seq_len + len(generated_tokens) - 1]], dtype=np.int64)

        attn_mask = np.ones((1, seq_len + len(generated_tokens)), dtype=np.int64)
        feed = {"input_ids": ids, "attention_mask": attn_mask, **cache}
        if use_position_ids:
            feed["position_ids"] = pos

        outputs = session.run(None, feed)
        logits = outputs[0][0, -1].copy()
        next_token = _sample_token(logits, generated_tokens)
        generated_tokens.append(next_token)

        for i, out in enumerate(session.get_outputs()[1:], 1):
            name = out.name.replace("present_conv", "past_conv").replace(
                "present.", "past_key_values."
            )
            if name in cache:
                cache[name] = outputs[i]

        if next_token == tokenizer.eos_token_id:
            break

    return tokenizer.decode(generated_tokens, skip_special_tokens=True)


print("Running sanity check...")
reply = generate([{"role": "user", "content": "Hi!"}])
print(f"Model says: {reply[:200]}")

## 4 — Register callback & launch Chat UI

In [ ]:
import json
from google.colab import output as colab_output
from IPython.display import display, HTML, JSON as IPJSON


def _chat_callback(messages_json):
    """Called from JS. Returns IPython JSON display object."""
    try:
        messages = json.loads(messages_json)
        reply = generate(messages)
        return IPJSON({"reply": reply})
    except Exception as e:
        return IPJSON({"error": str(e)})


colab_output.register_callback("notebook.chat", _chat_callback)


CHAT_HTML = """
<div id="lfm-chat" style="
  font-family: 'Segoe UI', system-ui, -apple-system, sans-serif;
  max-width: 640px;
  border: 1px solid #d0d0d0;
  border-radius: 12px;
  overflow: hidden;
  background: #fafafa;
">
  <div style="
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);
    color: white;
    padding: 14px 18px;
    font-size: 15px;
    font-weight: 600;
    display: flex;
    align-items: center;
    gap: 8px;
  ">
    <span style="font-size: 20px;">&#x1F4A7;</span>
    LFM2.5-350M &nbsp;<span style="font-weight:400; opacity:0.7; font-size:12px;">ONNX &middot; Q4</span>
  </div>

  <div id="lfm-messages" style="
    height: 380px;
    overflow-y: auto;
    padding: 16px;
    display: flex;
    flex-direction: column;
    gap: 10px;
  "></div>

  <div style="
    display: flex;
    border-top: 1px solid #e0e0e0;
    background: white;
    padding: 8px;
    gap: 8px;
  ">
    <input id="lfm-input" type="text" placeholder="Type a message..."
      style="
        flex: 1;
        border: 1px solid #d0d0d0;
        border-radius: 8px;
        padding: 10px 14px;
        font-size: 14px;
        outline: none;
      "
    />
    <button id="lfm-send" onclick="sendMessage()" style="
      background: #1a1a2e;
      color: white;
      border: none;
      border-radius: 8px;
      padding: 10px 18px;
      font-size: 14px;
      cursor: pointer;
      font-weight: 600;
    ">Send</button>
  </div>
</div>

<script>
(function() {
  const messagesDiv = document.getElementById("lfm-messages");
  const inputEl = document.getElementById("lfm-input");
  const sendBtn = document.getElementById("lfm-send");
  let history = [];

  function addBubble(role, text) {
    const isUser = role === "user";
    const bubble = document.createElement("div");
    bubble.style.cssText = `
      max-width: 82%;
      padding: 10px 14px;
      border-radius: 12px;
      font-size: 14px;
      line-height: 1.5;
      word-wrap: break-word;
      white-space: pre-wrap;
      align-self: ${isUser ? "flex-end" : "flex-start"};
      background: ${isUser ? "#1a1a2e" : "#e8e8e8"};
      color: ${isUser ? "#fff" : "#1a1a1a"};
    `;
    bubble.textContent = text;
    messagesDiv.appendChild(bubble);
    messagesDiv.scrollTop = messagesDiv.scrollHeight;
    return bubble;
  }

  // Extract the JSON payload from whatever format Colab returns
  function extractPayload(result) {
    // Try application/json first (IPython JSON display)
    if (result && result.data) {
      if (result.data["application/json"]) {
        const v = result.data["application/json"];
        return (typeof v === "string") ? JSON.parse(v) : v;
      }
      // Fallback: text/plain (plain string return)
      if (result.data["text/plain"]) {
        let t = result.data["text/plain"];
        // Python repr wraps strings in quotes: '"..."' or "'...'"
        if (typeof t === "string") {
          // Strip outer quotes from Python repr
          t = t.replace(/^['"]|['"]$/g, "");
          // Unescape
          t = t.replace(/\\'/g, "'").replace(/\\"/g, '"');
          return JSON.parse(t);
        }
      }
    }
    // Last resort: maybe result itself is the object
    if (result && result.reply) return result;
    throw new Error("Could not parse kernel response: " + JSON.stringify(result).substring(0, 200));
  }

  window.sendMessage = async function() {
    const text = inputEl.value.trim();
    if (!text) return;
    inputEl.value = "";
    inputEl.disabled = true;
    sendBtn.disabled = true;
    sendBtn.textContent = "...";

    addBubble("user", text);
    history.push({role: "user", content: text});
    const thinking = addBubble("assistant", "Thinking...");

    try {
      const result = await google.colab.kernel.invokeFunction(
        "notebook.chat",
        [JSON.stringify(history)],
        {}
      );
      const data = extractPayload(result);
      if (data.error) throw new Error(data.error);
      thinking.textContent = data.reply;
      history.push({role: "assistant", content: data.reply});
    } catch (err) {
      thinking.textContent = "Error: " + err.message;
      thinking.style.background = "#ffe0e0";
      thinking.style.color = "#900";
    }

    inputEl.disabled = false;
    sendBtn.disabled = false;
    sendBtn.textContent = "Send";
    inputEl.focus();
  };

  inputEl.addEventListener("keydown", (e) => {
    if (e.key === "Enter" && !e.shiftKey) { e.preventDefault(); sendMessage(); }
  });
})();
</script>
"""

display(HTML(CHAT_HTML))
print("Chat away! Each response may take 10-30s depending on length.")

---
### Troubleshooting
If you get parse errors, add a debug cell to inspect the raw return format:\n",
```python\n",
from google.colab import output\n",
from IPython.display import Javascript\n",
display(Javascript('''\n",
(async () => {\n",
  const r = await google.colab.kernel.invokeFunction("notebook.chat", ['{"role":"user","content":"test"}'], {});\n",
  console.log("RAW RESULT:", JSON.stringify(r));\n",
})();\n",
'''))\n",
```
Then check the browser console (F12) for the structure.

### Notes
- The model is **350M params (Q4, ~276MB)** — fast but limited quality vs larger models.
- Multi-turn conversation is supported — full chat history is sent each turn.
- To use **FP16** (~692MB, better quality), change `VARIANT = "model_fp16"` in cell 2.